In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")

In [6]:
train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [8]:
train.isnull().sum()

id                 0
Temparature        0
Humidity           0
Moisture           0
Soil Type          0
Crop Type          0
Nitrogen           0
Potassium          0
Phosphorous        0
Fertilizer Name    0
dtype: int64

In [10]:
test.isnull().sum()

id             0
Temparature    0
Humidity       0
Moisture       0
Soil Type      0
Crop Type      0
Nitrogen       0
Potassium      0
Phosphorous    0
dtype: int64

In [11]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               750000 non-null  int64 
 1   Temparature      750000 non-null  int64 
 2   Humidity         750000 non-null  int64 
 3   Moisture         750000 non-null  int64 
 4   Soil Type        750000 non-null  object
 5   Crop Type        750000 non-null  object
 6   Nitrogen         750000 non-null  int64 
 7   Potassium        750000 non-null  int64 
 8   Phosphorous      750000 non-null  int64 
 9   Fertilizer Name  750000 non-null  object
dtypes: int64(7), object(3)
memory usage: 57.2+ MB


In [4]:
from sklearn.preprocessing import LabelEncoder

cat_cols = train.select_dtypes(include=["object"]).columns

for c in cat_cols:
  le = LabelEncoder()
  le.fit(train[c])
  if c == "Fertilizer Name":
    train[c] = le.transform(train[c])
  else:
    train[c] = le.transform(train[c])
    test[c] = le.transform(test[c])

In [19]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

cat_cols = ["Soil Type", "Crop Type"]
target_cols = "Fertilizer Name"

le = LabelEncoder()
le.fit(train[target_cols])
train[target_cols] = le.transform(train[target_cols])

for c in cat_cols:
  data_tmp = pd.DataFrame({c: train[c], "target": train[target_cols]})
  target_mean = data_tmp.groupby(c)["target"].mean()
  
  test[c] = test[c].map(target_mean)
  
  tmp = np.repeat(np.nan, train.shape[0])
  
  kf = KFold(n_splits=4, shuffle=True, random_state=42)
  for idx_1, idx_2 in kf.split(train):
    target_mean = data_tmp.iloc[idx_1].groupby(c)["target"].mean()
    tmp[idx_2] = train[c].iloc[idx_2].map(target_mean)
  
  train[c] = tmp

In [20]:
train_x = train.drop(["id", "Fertilizer Name"], axis=1)
train_y = train["Fertilizer Name"]
test_x = test.drop(["id"], axis=1)

In [24]:
train.nunique()

id                 750000
Temparature            14
Humidity               23
Moisture               41
Soil Type               5
Crop Type              11
Nitrogen               39
Potassium              20
Phosphorous            43
Fertilizer Name         7
dtype: int64

In [27]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report


X_train, X_valid, y_train, y_valid = train_test_split(train_x, train_y, test_size=0.3, shuffle=True)

model = XGBClassifier()

model.fit(X_train, y_train)

y_valid_pred_pre = model.predict(X_valid)

y_valid_pred = pd.Series(np.array(y_valid_pred_pre).T)

df_y = pd.concat([y_valid.reset_index().drop(["index"], axis=1), y_valid_pred], axis=1)

df_y = df_y.rename(columns={"Fertilizer Name": "y_valid", 0: "y_valid_pred"})

acc = df_y[df_y["y_valid"] == df_y["y_valid_pred"]].shape[0] / df_y.shape[0] * 100


print(acc)


KeyError: 'y_valid'